Step 1: Load match_info and sort all matches chronologically.

In [1]:
import pandas as pd
import numpy as np

match_info = pd.read_csv('data/processed/match_info.csv')
match_info['date'] = pd.to_datetime(match_info['date'], format='mixed')
match_info = match_info.sort_values('date').reset_index(drop=True)

print(match_info.shape)
match_info[['match_id', 'date', 'season', 'team1', 'team2', 'winner']].head(10)

(1243, 14)


,match_id,date,season,team1,team2,winner
0,335982,2008-04-18,2007/08,Royal Challengers Bengaluru,Kolkata Knight Riders,Kolkata Knight Riders
1,335983,2008-04-19,2007/08,Punjab Kings,Chennai Super Kings,Chennai Super Kings
2,335984,2008-04-19,2007/08,Delhi Capitals,Rajasthan Royals,Delhi Capitals
3,335986,2008-04-20,2007/08,Kolkata Knight Riders,Deccan Chargers,Kolkata Knight Riders
4,335985,2008-04-20,2007/08,Mumbai Indians,Royal Challengers Bengaluru,Royal Challengers Bengaluru
5,335987,2008-04-21,2007/08,Rajasthan Royals,Punjab Kings,Rajasthan Royals
6,335988,2008-04-22,2007/08,Deccan Chargers,Delhi Capitals,Delhi Capitals
7,335989,2008-04-23,2007/08,Chennai Super Kings,Mumbai Indians,Chennai Super Kings
8,335990,2008-04-24,2007/08,Deccan Chargers,Rajasthan Royals,Rajasthan Royals
9,335991,2008-04-25,2007/08,Punjab Kings,Mumbai Indians,Punjab Kings


Step 2: Create the team1_won target label, dropping matches with no result.

In [2]:
# drop matches with no winner (ties/no-results) - can't train on an undefined outcome
match_info_clean = match_info[match_info['winner'].notnull()].copy()

match_info_clean['team1_won'] = (match_info_clean['winner'] == match_info_clean['team1']).astype(int)

print(match_info_clean.shape)
print(match_info_clean['team1_won'].value_counts())

(1218, 15)
team1_won
0    610
1    608
Name: count, dtype: int64


Step 3: Compute each team's leak-free prior win rate (uses only matches before the current one).

In [3]:
def compute_prior_win_rate(df):
    """For each match, calculate each team's win % using ONLY prior matches."""
    team_stats = {}  # {team_name: [wins, total_matches]}
    team1_win_rate = []
    team2_win_rate = []

    for _, row in df.iterrows():
        t1, t2 = row['team1'], row['team2']

        # get each team's record BEFORE this match (default 0.5 if no history yet)
        t1_wins, t1_total = team_stats.get(t1, [0, 0])
        t2_wins, t2_total = team_stats.get(t2, [0, 0])

        team1_win_rate.append(t1_wins / t1_total if t1_total > 0 else 0.5)
        team2_win_rate.append(t2_wins / t2_total if t2_total > 0 else 0.5)

        # NOW update the stats with this match's result (only after recording the "before" state)
        for team in [t1, t2]:
            wins, total = team_stats.get(team, [0, 0])
            team_stats[team] = [wins + (1 if row['winner'] == team else 0), total + 1]

    return team1_win_rate, team2_win_rate

match_info_clean = match_info_clean.sort_values('date').reset_index(drop=True)
t1_wr, t2_wr = compute_prior_win_rate(match_info_clean)
match_info_clean['team1_win_rate'] = t1_wr
match_info_clean['team2_win_rate'] = t2_wr

match_info_clean[['date', 'team1', 'team2', 'team1_win_rate', 'team2_win_rate', 'winner']].head(15)

,date,team1,team2,team1_win_rate,team2_win_rate,winner
0,2008-04-18,Royal Challengers Bengaluru,Kolkata Knight Riders,0.500000,0.500000,Kolkata Knight Riders
1,2008-04-19,Punjab Kings,Chennai Super Kings,0.500000,0.500000,Chennai Super Kings
2,2008-04-19,Delhi Capitals,Rajasthan Royals,0.500000,0.500000,Delhi Capitals
3,2008-04-20,Kolkata Knight Riders,Deccan Chargers,1.000000,0.500000,Kolkata Knight Riders
4,2008-04-20,Mumbai Indians,Royal Challengers Bengaluru,0.500000,0.000000,Royal Challengers Bengaluru
5,2008-04-21,Rajasthan Royals,Punjab Kings,0.000000,0.000000,Rajasthan Royals
6,2008-04-22,Deccan Chargers,Delhi Capitals,0.000000,1.000000,Delhi Capitals
7,2008-04-23,Chennai Super Kings,Mumbai Indians,1.000000,0.000000,Chennai Super Kings
8,2008-04-24,Deccan Chargers,Rajasthan Royals,0.000000,0.500000,Rajasthan Royals
9,2008-04-25,Punjab Kings,Mumbai Indians,0.000000,0.000000,Punjab Kings


Step 4: Compute each team's leak-free recent form over their last 5 matches.

In [4]:
def compute_recent_form(df, window=5):
    team_history = {}  # {team_name: [1, 0, 1, ...]} list of recent results
    team1_form = []
    team2_form = []

    for _, row in df.iterrows():
        t1, t2 = row['team1'], row['team2']

        h1 = team_history.get(t1, [])
        h2 = team_history.get(t2, [])
        team1_form.append(np.mean(h1[-window:]) if h1 else 0.5)
        team2_form.append(np.mean(h2[-window:]) if h2 else 0.5)

        for team in [t1, t2]:
            hist = team_history.get(team, [])
            hist.append(1 if row['winner'] == team else 0)
            team_history[team] = hist

    return team1_form, team2_form

t1_form, t2_form = compute_recent_form(match_info_clean)
match_info_clean['team1_recent_form'] = t1_form
match_info_clean['team2_recent_form'] = t2_form

match_info_clean[['date', 'team1', 'team2', 'team1_recent_form', 'team2_recent_form']].tail(10)

,date,team1,team2,team1_recent_form,team2_recent_form
1208,2026-05-20,Mumbai Indians,Kolkata Knight Riders,0.4,0.8
1209,2026-05-21,Gujarat Titans,Chennai Super Kings,0.8,0.6
1210,2026-05-22,Sunrisers Hyderabad,Royal Challengers Bengaluru,0.6,0.6
1211,2026-05-23,Lucknow Super Giants,Punjab Kings,0.4,0.0
1212,2026-05-24,Rajasthan Royals,Mumbai Indians,0.4,0.4
1213,2026-05-24,Delhi Capitals,Kolkata Knight Riders,0.6,0.8
1214,2026-05-26,Royal Challengers Bengaluru,Gujarat Titans,0.6,0.8
1215,2026-05-27,Rajasthan Royals,Sunrisers Hyderabad,0.4,0.6
1216,2026-05-29,Rajasthan Royals,Gujarat Titans,0.6,0.6
1217,2026-05-31,Gujarat Titans,Royal Challengers Bengaluru,0.6,0.8


Step 5: Compute each team's leak-free venue-specific win rate.

In [5]:
def compute_venue_win_rate(df):
    venue_team_stats = {}  # {(venue, team): [wins, total]}
    team1_venue_wr = []
    team2_venue_wr = []

    for _, row in df.iterrows():
        venue, t1, t2 = row['venue'], row['team1'], row['team2']

        w1, n1 = venue_team_stats.get((venue, t1), [0, 0])
        w2, n2 = venue_team_stats.get((venue, t2), [0, 0])
        team1_venue_wr.append(w1 / n1 if n1 > 0 else 0.5)
        team2_venue_wr.append(w2 / n2 if n2 > 0 else 0.5)

        for team in [t1, t2]:
            w, n = venue_team_stats.get((venue, team), [0, 0])
            venue_team_stats[(venue, team)] = [w + (1 if row['winner'] == team else 0), n + 1]

    return team1_venue_wr, team2_venue_wr

t1_vwr, t2_vwr = compute_venue_win_rate(match_info_clean)
match_info_clean['team1_venue_win_rate'] = t1_vwr
match_info_clean['team2_venue_win_rate'] = t2_vwr

match_info_clean[['date', 'venue', 'team1', 'team2', 'team1_venue_win_rate', 'team2_venue_win_rate']].tail(10)

,date,venue,team1,team2,team1_venue_win_rate,team2_venue_win_rate
1208,2026-05-20,"""Eden Gardens, Kolkata""",Mumbai Indians,Kolkata Knight Riders,0.000000,0.458333
1209,2026-05-21,"""Narendra Modi Stadium, Ahmedabad""",Gujarat Titans,Chennai Super Kings,0.586207,0.500000
1210,2026-05-22,"""Rajiv Gandhi International Stadium, Uppal, Hy...",Sunrisers Hyderabad,Royal Challengers Bengaluru,0.500000,1.000000
1211,2026-05-23,"""Bharat Ratna Shri Atal Bihari Vajpayee Ekana ...",Lucknow Super Giants,Punjab Kings,0.440000,0.666667
1212,2026-05-24,"""Wankhede Stadium, Mumbai""",Rajasthan Royals,Mumbai Indians,0.500000,0.548387
1213,2026-05-24,"""Eden Gardens, Kolkata""",Delhi Capitals,Kolkata Knight Riders,0.000000,0.480000
1214,2026-05-26,"""Himachal Pradesh Cricket Association Stadium,...",Royal Challengers Bengaluru,Gujarat Titans,1.000000,0.500000
1215,2026-05-27,"""Maharaja Yadavindra Singh International Crick...",Rajasthan Royals,Sunrisers Hyderabad,1.000000,0.000000
1216,2026-05-29,"""Maharaja Yadavindra Singh International Crick...",Rajasthan Royals,Gujarat Titans,1.000000,0.000000
1217,2026-05-31,"""Narendra Modi Stadium, Ahmedabad""",Gujarat Titans,Royal Challengers Bengaluru,0.600000,0.428571


Step 6: Add toss-based features (no leakage risk — known before the match starts).

In [6]:
match_info_clean['toss_won_by_team1'] = (match_info_clean['toss_winner'] == match_info_clean['team1']).astype(int)
match_info_clean['team1_bat_first'] = (
    (match_info_clean['toss_won_by_team1'] == 1) & (match_info_clean['toss_decision'] == 'bat')
).astype(int)

Step 7: Train a first Logistic Regression winner-prediction model.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss

feature_cols = [
    'team1_win_rate', 'team2_win_rate',
    'team1_recent_form', 'team2_recent_form',
    'team1_venue_win_rate', 'team2_venue_win_rate',
    'toss_won_by_team1', 'team1_bat_first'
]

X = match_info_clean[feature_cols]
y = match_info_clean['team1_won']

print(X.shape, y.shape)
print(X.isnull().sum())

(1218, 8) (1218,)
team1_win_rate          0
team2_win_rate          0
team1_recent_form       0
team2_recent_form       0
team1_venue_win_rate    0
team2_venue_win_rate    0
toss_won_by_team1       0
team1_bat_first         0
dtype: int64


Step 8: Create a time-aware chronological train/test split.

In [8]:
split_index = int(len(match_info_clean) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Train: {X_train.shape[0]} matches (up to {match_info_clean.iloc[split_index-1]['date']})")
print(f"Test: {X_test.shape[0]} matches (from {match_info_clean.iloc[split_index]['date']})")

Train: 974 matches (up to 2023-04-30 00:00:00)
Test: 244 matches (from 2023-05-01 00:00:00)


Step 9: Train Logistic Regression and evaluate with multiple classification metrics.

In [9]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))
print("Log Loss:", log_loss(y_test, y_pred_proba))

Accuracy: 0.5
Precision: 0.460431654676259
Recall: 0.5765765765765766
F1: 0.512
ROC-AUC: 0.5011853959222381
Log Loss: 0.6971237156833981


Step 10: Diagnose weak features by correlating them directly with the match outcome.

In [10]:
# does higher win_rate difference actually associate with winning?
match_info_clean['win_rate_diff'] = match_info_clean['team1_win_rate'] - match_info_clean['team2_win_rate']
print(match_info_clean.groupby('team1_won')['win_rate_diff'].mean())

# correlation of each feature with the outcome
print(X.corrwith(y).sort_values())

team1_won
0   -0.013526
1    0.004386
Name: win_rate_diff, dtype: float64
team2_win_rate         -0.017807
team2_venue_win_rate    0.000514
team1_venue_win_rate    0.005566
team1_bat_first         0.007146
team2_recent_form       0.025737
toss_won_by_team1       0.030884
team1_recent_form       0.051915
team1_win_rate          0.070325
dtype: float64


Step 11: Compute leak-free head-to-head win rate between each team pairing.

In [11]:
def compute_h2h_win_rate(df):
    h2h_stats = {}  # {(team_a, team_b): [team_a_wins, total]}
    h2h_wr = []

    for _, row in df.iterrows():
        t1, t2 = row['team1'], row['team2']
        key = tuple(sorted([t1, t2]))  # order-independent pairing
        wins, total = h2h_stats.get(key, [0, 0])

        # win rate FOR team1 specifically in this matchup
        if total == 0:
            h2h_wr.append(0.5)
        else:
            t1_prior_wins = wins if key[0] == t1 else (total - wins)
            h2h_wr.append(t1_prior_wins / total)

        # update: track wins for key[0] (alphabetically first team in the pair)
        winner = row['winner']
        wins_for_key0 = wins + (1 if winner == key[0] else 0)
        h2h_stats[key] = [wins_for_key0, total + 1]

    return h2h_wr

match_info_clean['h2h_win_rate_team1'] = compute_h2h_win_rate(match_info_clean)

Step 12: Compute leak-free batting strength from each team's recent innings scores.

In [12]:
all_matches = pd.read_csv('data/processed/all_matches_clean.csv')
all_matches['total_runs'] = all_matches['runs_off_bat'] + all_matches['extras']

innings_totals = all_matches[all_matches['innings'].isin([1,2])].groupby(
    ['match_id', 'batting_team']
)['total_runs'].sum().reset_index()

innings_totals = innings_totals.merge(
    match_info_clean[['match_id', 'date']], on='match_id'
).sort_values('date')

def compute_batting_strength(match_df, innings_df, window=5):
    team_scores = {}
    t1_strength, t2_strength = [], []

    for _, row in match_df.iterrows():
        t1, t2 = row['team1'], row['team2']
        h1 = team_scores.get(t1, [])
        h2 = team_scores.get(t2, [])
        t1_strength.append(np.mean(h1[-window:]) if h1 else 160)  # league-average default
        t2_strength.append(np.mean(h2[-window:]) if h2 else 160)

        this_match_scores = innings_df[innings_df['match_id'] == row['match_id']]
        for _, s in this_match_scores.iterrows():
            team = s['batting_team']
            hist = team_scores.get(team, [])
            hist.append(s['total_runs'])
            team_scores[team] = hist

    return t1_strength, t2_strength

t1_bs, t2_bs = compute_batting_strength(match_info_clean, innings_totals)
match_info_clean['team1_batting_strength'] = t1_bs
match_info_clean['team2_batting_strength'] = t2_bs

C:\Users\ramha.LAPTOP-AB00D4P2\AppData\Local\Temp\ipykernel_11072\2074517764.py:1: DtypeWarning: Columns (0: season, 1: non_boundary, 2: fielder_3) have mixed types. Specify dtype option on import or set low_memory=False.
  all_matches = pd.read_csv('data/processed/all_matches_clean.csv')


Step 13: Check correlation of the new head-to-head and batting-strength features.

In [13]:
match_info_clean['batting_diff'] = match_info_clean['team1_batting_strength'] - match_info_clean['team2_batting_strength']
print(match_info_clean[['h2h_win_rate_team1', 'batting_diff']].corrwith(match_info_clean['team1_won']))

h2h_win_rate_team1    0.019158
batting_diff          0.027972
dtype: float64


Step 14: Compute leak-free bowling strength and save the full feature table.

In [14]:
bowling_totals = all_matches[all_matches['innings'].isin([1,2])].groupby(
    ['match_id', 'bowling_team']
)['total_runs'].sum().reset_index()

bowling_totals = bowling_totals.merge(
    match_info_clean[['match_id', 'date']], on='match_id'
).sort_values('date')

def compute_bowling_strength(match_df, bowling_df, window=5):
    team_conceded = {}
    t1_bowl, t2_bowl = [], []

    for _, row in match_df.iterrows():
        t1, t2 = row['team1'], row['team2']
        h1 = team_conceded.get(t1, [])
        h2 = team_conceded.get(t2, [])
        t1_bowl.append(np.mean(h1[-window:]) if h1 else 160)
        t2_bowl.append(np.mean(h2[-window:]) if h2 else 160)

        this_match = bowling_df[bowling_df['match_id'] == row['match_id']]
        for _, s in this_match.iterrows():
            team = s['bowling_team']
            hist = team_conceded.get(team, [])
            hist.append(s['total_runs'])
            team_conceded[team] = hist

    return t1_bowl, t2_bowl

t1_bowl, t2_bowl = compute_bowling_strength(match_info_clean, bowling_totals)
match_info_clean['team1_bowling_strength'] = t1_bowl
match_info_clean['team2_bowling_strength'] = t2_bowl

# lower runs conceded = better bowling, so flip sign for intuitive "strength"
match_info_clean.to_csv('data/processed/match_features.csv', index=False)
print("Saved:", match_info_clean.shape)

Saved: (1218, 30)


Step 15: Retrain Logistic Regression and Random Forest on the full feature set.

In [15]:
from sklearn.ensemble import RandomForestClassifier

feature_cols = [
    'team1_win_rate', 'team2_win_rate',
    'team1_recent_form', 'team2_recent_form',
    'team1_venue_win_rate', 'team2_venue_win_rate',
    'toss_won_by_team1', 'team1_bat_first',
    'h2h_win_rate_team1',
    'team1_batting_strength', 'team2_batting_strength',
    'team1_bowling_strength', 'team2_bowling_strength'
]

X = match_info_clean[feature_cols]
y = match_info_clean['team1_won']

split_index = int(len(match_info_clean) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# Logistic Regression, full feature set
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train, y_train)
log_proba = log_model.predict_proba(X_test)[:, 1]
print("Logistic Regression — Accuracy:", accuracy_score(y_test, log_model.predict(X_test)), "| ROC-AUC:", roc_auc_score(y_test, log_proba))

# Random Forest, same features — can catch interactions Logistic Regression can't
rf_model = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)
rf_proba = rf_model.predict_proba(X_test)[:, 1]
print("Random Forest — Accuracy:", accuracy_score(y_test, rf_model.predict(X_test)), "| ROC-AUC:", roc_auc_score(y_test, rf_proba))

Logistic Regression — Accuracy: 0.5 | ROC-AUC: 0.49800176115965583
Random Forest — Accuracy: 0.5450819672131147 | ROC-AUC: 0.5300413195150038


Step 16: Compare model accuracy against majority-class and toss-only baselines.

In [16]:
# baseline 1: what if we always predicted team1 wins?
print("Majority-class baseline:", max(y_test.mean(), 1 - y_test.mean()))

# baseline 2: does simply winning the toss predict the match winner? (a classic cricket question, and your spec asks this directly)
toss_baseline = (match_info_clean.iloc[split_index:]['toss_won_by_team1'] == match_info_clean.iloc[split_index:]['team1_won']).mean()
print("Toss-only baseline accuracy:", toss_baseline)

Majority-class baseline: 0.5450819672131147
Toss-only baseline accuracy: 0.5163934426229508


Step 17: Train Gradient Boosting and inspect feature importances.

In [17]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
gb_model.fit(X_train, y_train)
gb_proba = gb_model.predict_proba(X_test)[:, 1]
print("Gradient Boosting — Accuracy:", accuracy_score(y_test, gb_model.predict(X_test)), "| ROC-AUC:", roc_auc_score(y_test, gb_proba))

# which features is Random Forest actually relying on?
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)

Gradient Boosting — Accuracy: 0.5245901639344263 | ROC-AUC: 0.5145973040709882
team2_bowling_strength    0.144780
team1_batting_strength    0.129683
team1_bowling_strength    0.124479
team2_batting_strength    0.122837
team2_win_rate            0.108686
team1_win_rate            0.106423
team1_venue_win_rate      0.068319
h2h_win_rate_team1        0.064455
team1_recent_form         0.038733
team2_venue_win_rate      0.038635
team2_recent_form         0.034308
toss_won_by_team1         0.009620
team1_bat_first           0.009042
dtype: float64


Step 18: Diagnose the models' prediction distributions against actual outcomes.

In [18]:
print("RF prediction distribution:", pd.Series(rf_model.predict(X_test)).value_counts())
print("GB prediction distribution:", pd.Series(gb_model.predict(X_test)).value_counts())
print("Actual distribution:", y_test.value_counts())

RF prediction distribution: 0    156
1     88
Name: count, dtype: int64
GB prediction distribution: 0    149
1     95
Name: count, dtype: int64
Actual distribution: team1_won
0    133
1    111
Name: count, dtype: int64


Step 1: Build the in-match snapshot dataset

In [19]:
all_matches = pd.read_csv('data/processed/all_matches_clean.csv')
all_matches['season'] = all_matches['season'].astype(str)
all_matches['total_runs'] = all_matches['runs_off_bat'] + all_matches['extras']
all_matches['is_wicket'] = all_matches['wicket_type'].notnull().astype(int)
all_matches['over_num'] = all_matches['ball'].astype(str).str.split('.').str[0].astype(int)

main_deliveries = all_matches[all_matches['innings'].isin([1, 2])].copy()
main_deliveries = main_deliveries.sort_values(['match_id', 'innings', 'delivery_seq'])

# running totals within each innings, ball by ball
main_deliveries['current_score'] = main_deliveries.groupby(['match_id', 'innings'])['total_runs'].cumsum()
main_deliveries['current_wickets'] = main_deliveries.groupby(['match_id', 'innings'])['is_wicket'].cumsum()
main_deliveries['balls_so_far'] = main_deliveries.groupby(['match_id', 'innings']).cumcount() + 1

# the actual FINAL score of that innings (our prediction target)
final_scores = main_deliveries.groupby(['match_id', 'innings'])['total_runs'].sum().reset_index()
final_scores.columns = ['match_id', 'innings', 'final_score']

main_deliveries = main_deliveries.merge(final_scores, on=['match_id', 'innings'])

print(main_deliveries.shape)
main_deliveries[['match_id', 'innings', 'over_num', 'balls_so_far', 'current_score', 'current_wickets', 'final_score']].head(15)

C:\Users\ramha.LAPTOP-AB00D4P2\AppData\Local\Temp\ipykernel_11072\3986366264.py:1: DtypeWarning: Columns (0: season, 1: non_boundary, 2: fielder_3) have mixed types. Specify dtype option on import or set low_memory=False.
  all_matches = pd.read_csv('data/processed/all_matches_clean.csv')


(295557, 35)


,match_id,innings,over_num,balls_so_far,current_score,current_wickets,final_score
0,335982,1,0,1,1,0,222
1,335982,1,0,2,1,0,222
2,335982,1,0,3,2,0,222
3,335982,1,0,4,2,0,222
4,335982,1,0,5,2,0,222
5,335982,1,0,6,2,0,222
6,335982,1,0,7,3,0,222
7,335982,1,1,8,3,0,222
8,335982,1,1,9,7,0,222
9,335982,1,1,10,11,0,222


Step 2: Build prediction features + filter to realistic prediction points

In [20]:
main_deliveries['current_run_rate'] = (main_deliveries['current_score'] / main_deliveries['balls_so_far']) * 6
main_deliveries['balls_remaining'] = 120 - main_deliveries['balls_so_far']  # 20 overs = 120 balls

# runs scored in the last 24 balls (~4 overs) - captures recent momentum, not just the whole-innings average
main_deliveries['runs_last_24balls'] = main_deliveries.groupby(['match_id', 'innings'])['total_runs'].transform(
    lambda x: x.rolling(window=24, min_periods=1).sum()
)

# only keep snapshots from over 5 onwards (30+ balls bowled) - enough info to make a meaningful prediction
score_data = main_deliveries[main_deliveries['balls_so_far'] >= 30].copy()

print(score_data.shape)
score_data[['match_id', 'innings', 'balls_so_far', 'current_score', 'current_wickets',
            'current_run_rate', 'runs_last_24balls', 'balls_remaining', 'final_score']].sample(10, random_state=42)

(223697, 38)


,match_id,innings,balls_so_far,current_score,current_wickets,current_run_rate,runs_last_24balls,balls_remaining,final_score
294039,1529311,2,65,107,2,9.876923,42.0,55,200
197436,1254074,1,69,64,2,5.565217,30.0,51,131
195057,1254064,1,102,112,6,6.588235,24.0,18,147
80165,598013,2,84,98,3,7.000000,31.0,36,132
245372,1422125,2,101,119,6,7.069307,22.0,19,143
291338,1529300,2,64,99,1,9.281250,31.0,56,194
239744,1359532,2,81,85,2,6.296296,22.0,39,185
81125,598017,2,70,98,2,8.400000,40.0,50,152
53258,501248,1,59,70,2,7.118644,36.0,61,119
121193,829813,1,85,137,2,9.670588,48.0,35,187


Step 3: Time-aware split + train regression models

In [21]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# bring in match date to sort chronologically, same leak-safe approach as Phase 4A
score_data = score_data.merge(match_info[['match_id', 'date']], on='match_id')
score_data = score_data.sort_values('date').reset_index(drop=True)

feature_cols_score = ['current_score', 'current_wickets', 'current_run_rate',
                       'runs_last_24balls', 'balls_remaining']

X = score_data[feature_cols_score]
y = score_data['final_score']

split_index = int(len(score_data) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

Train: 178957 | Test: 44740


Step 22: Train and compare Linear Regression, Random Forest, and Gradient Boosting.

In [22]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
}

for name, m in models.items():
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    r2 = r2_score(y_test, preds)
    print(f"{name} — MAE: {mae:.2f} | RMSE: {rmse:.2f} | R²: {r2:.3f}")

Linear Regression — MAE: 18.49 | RMSE: 24.34 | R²: 0.511


Random Forest — MAE: 19.04 | RMSE: 25.56 | R²: 0.460
Gradient Boosting — MAE: 18.85 | RMSE: 25.20 | R²: 0.476


Step 23: Compare all three regression models against a naive run-rate baseline.

In [23]:
naive_pred = X_test['current_score'] + (X_test['current_run_rate'] * X_test['balls_remaining'] / 6)
naive_mae = mean_absolute_error(y_test, naive_pred)
print(f"Naive baseline (extrapolate current run rate) — MAE: {naive_mae:.2f}")

Naive baseline (extrapolate current run rate) — MAE: 21.52


Check if predictions get better later in the innings (makes intuitive sense — less uncertainty near the end)

In [24]:
score_data_test = score_data.iloc[split_index:].copy()
score_data_test['pred'] = models['Linear Regression'].predict(X_test)
score_data_test['abs_error'] = abs(score_data_test['pred'] - score_data_test['final_score'])

# error by over stage
score_data_test['over_stage'] = pd.cut(score_data_test['balls_so_far'], 
                                          bins=[29, 60, 90, 120], 
                                          labels=['Overs 5-10', 'Overs 10-15', 'Overs 15-20'])
print(score_data_test.groupby('over_stage', observed=True)['abs_error'].mean())

over_stage
Overs 5-10     25.551551
Overs 10-15    19.252373
Overs 15-20    11.314814
Name: abs_error, dtype: float64


Step 25: Save the trained score-prediction model and feature table to disk.

In [25]:
import joblib
import os

os.makedirs('models', exist_ok=True)
joblib.dump(models['Linear Regression'], 'models/score_model.pkl')

score_data.to_csv('data/processed/score_features.csv', index=False)
print("Model and data saved.")

Model and data saved.
